[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/19_containers_docker/lab01_container_anatomy.ipynb)

# 🧪 Lab 1 — Container anatomy: build Docker's guts in pure Python

> **Module:** Containers & Docker (Module 19) · **Estimated time:** ~60 minutes · **Difficulty:** Intermediate

This lab is the hands-on companion to the [Images & layers](images-and-layers.md) and [Container internals](container-internals.md) chapters. Those chapters *describe* the machinery — content-addressed layers, copy-on-write, OverlayFS, namespaces, cgroups. Here you **build a working model of each piece in plain Python**, small enough to read in one sitting, faithful enough that every behavior you observe is a behavior real Docker has.

**Nothing in this lab needs Docker, an internet connection, or admin rights** — it is 100% offline, standard-library Python (plus pandas for pretty tables). Every toy is grounded in the real command it mirrors:

| You build (pure Python) | The real thing it mirrors |
|---|---|
| A content-addressed blob store + mutable tags | Registry/`/var/lib/docker` storage · `docker images --digests` |
| Layers as file-tree diffs with whiteouts | Layer tarballs · `docker history` · `.wh.` entries |
| A build-cache replay with invalidation cascade | `docker build` layer caching |
| An overlay lookup (upper/lower dirs, copy-up) | OverlayFS `lowerdir`/`upperdir`/`merged` mounts |
| Isolated process tables (pid-namespace objects) | `unshare --pid` · `docker top` · PID 1 |
| A memory-metering cgroup that OOM-kills | `docker run --memory` · exit code 137 |

Work top to bottom. ✋ checkpoints ask you to write a few lines before the solution; 🧪 practice exercises and a 🎁 mini-project close the lab.

## 🎯 Learning objectives

By the end of this lab you can:

1. **Explain content addressing** — why an image artifact's identity *is* the SHA-256 hash of its bytes, and why tags are just mutable pointers onto that store.
2. **Model image layers as diffs** — additions plus whiteouts — and show why a file deleted in a later layer still costs storage and bandwidth.
3. **Replay the build cache** — compute per-instruction cache keys (parent + instruction + copied-file content) and predict exactly which layers a given edit rebuilds.
4. **Implement the OverlayFS lookup** — upper-wins/top-down search, copy-up on modification, whiteouts on deletion — and account for what lives in the container layer vs the shared image.
5. **Simulate pid-namespace isolation** — two containers that are each PID 1, a host that sees both under real PIDs, and `docker exec` as *joining* a namespace.
6. **Simulate a cgroup memory limit** — and recognize the `OOMKilled` / exit-137 fingerprint from [container-internals.md §4](container-internals.md).

## 1. Content addressing — every artifact is its hash

[Images-and-layers §4](images-and-layers.md) makes the claim that the whole image system stands on one idea: **store every artifact under the SHA-256 hash of its bytes**. Same hash ⟹ same bytes, mathematically — so identity, verification, and deduplication all come for free, exactly like Git commits.

On a real machine you'd see these identities with:

```bash
docker images --digests            # the sha256:… manifest digest per image
docker image inspect python:3.12-slim   # "Id": the config blob's digest
```

Let's build the primitive. One helper turns any JSON-serializable object into its **digest** (we keep 12 hex chars for readability — real Docker shows 64):

In [1]:
import hashlib, json


def digest(obj) -> str:
    """Content id of an object: sha256 over its canonical JSON bytes."""
    payload = json.dumps(obj, sort_keys=True).encode()
    return "sha256:" + hashlib.sha256(payload).hexdigest()[:12]


file_v1 = {"requirements.txt": "fastapi==0.111\nuvicorn==0.30\n"}

print("digest:        ", digest(file_v1))
print("same bytes:    ", digest({"requirements.txt": "fastapi==0.111\nuvicorn==0.30\n"}))
print("1 char changed:", digest({"requirements.txt": "fastapi==0.112\nuvicorn==0.30\n"}))

digest:         sha256:06eedb5a1bbf
same bytes:     sha256:06eedb5a1bbf
1 char changed: sha256:b2b29941d52e


Identical content → identical digest, every time, on every machine. One changed character → an unrelated digest. That determinism is what lets a registry in Frankfurt and a laptop in Lisbon agree they hold *the same layer* without exchanging the bytes.

Now the two structures the whole lab runs on:

- **`BLOBS`** — the content-addressed store: `digest → object`. This models both a registry's blob storage and your local `/var/lib/docker`.
- **`TAGS`** — the mutable name layer: `"name:tag" → digest`. This is the part humans see — and the part that can be silently re-pointed, which is the entire tags-vs-digests lesson.

In [2]:
BLOBS: dict[str, object] = {}   # digest -> stored object   (content-addressed, append-only)
TAGS:  dict[str, str]    = {}   # "name:tag" -> digest      (mutable pointers)


def put(obj) -> str:
    """Store an object by its content digest (idempotent — content dedups itself)."""
    d = digest(obj)
    BLOBS[d] = obj
    return d


d1 = put({"requirements.txt": "fastapi==0.111\n"})
d2 = put({"requirements.txt": "fastapi==0.112\n"})

TAGS["deps:latest"] = d1
print("deps:latest ->", TAGS["deps:latest"])

TAGS["deps:latest"] = d2                      # someone pushed a new version...
print("deps:latest ->", TAGS["deps:latest"], "   <- same NAME, different bytes!")
print("but", d1, "still exists and is still exactly:", BLOBS[d1])

deps:latest -> sha256:76a49aa3073e
deps:latest -> sha256:2e297ba78ad5    <- same NAME, different bytes!
but sha256:76a49aa3073e still exists and is still exactly: {'requirements.txt': 'fastapi==0.111\n'}


---

### ✋ Quick exercise (~2 min) — Resolve like a registry

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Write `resolve(ref)` that accepts **either** kind of reference — a tag like `"deps:latest"` or a digest like `"sha256:…"` — and returns the stored object (or `None` if unknown). This mirrors what happens when you `docker pull name:tag` vs `docker pull name@sha256:…`.

1. `resolve("deps:latest")` should return the *current* pointee.
2. `resolve(d1)` should still return the old object — content addressing never forgets.

In [3]:
# ✍️ Your turn 👇
def resolve(ref: str):
    # digests start with "sha256:" and hit BLOBS directly;
    # anything else is a tag -> look up its digest first
    ...

# print(resolve("deps:latest"))
# print(resolve(d1))
# print(resolve("nope:missing"))

<details>
<summary>✅ <b>Solution</b></summary>

```python
def resolve(ref: str):
    if ref.startswith("sha256:"):
        return BLOBS.get(ref)
    d = TAGS.get(ref)
    return BLOBS.get(d) if d else None


print(resolve("deps:latest"))   # the v2 object — tags follow the pointer
print(resolve(d1))              # the v1 object — digests are forever
print(resolve("nope:missing"))  # None
```

Two lookup paths, two trust models: a tag answers *"whatever this name currently means"*, a digest answers *"these exact bytes."* Production deploys and reproducible research pin digests for exactly this reason ([images-and-layers.md §4](images-and-layers.md)).
</details>

## 2. Layers — filesystem diffs, with whiteouts

An image layer is **not** a snapshot of the whole filesystem — it's a **diff**: files added or changed at this step, plus **whiteouts** (markers meaning "the file below is now deleted"). In a real layer tarball a whiteout is an entry named `.wh.<filename>`; you can see per-layer sizes with:

```bash
docker history python:3.12-slim     # one row per layer; 0B rows are metadata-only
```

We model a layer as `{adds: {path: content}, deletes: [paths]}`, stored — like everything — in `BLOBS` by digest. Let's hand-build the layers of a small Python app image:

In [4]:
import pandas as pd


def make_layer(adds: dict | None = None, deletes: set | frozenset = frozenset()) -> str:
    """A layer = a filesystem diff. Returns its content digest."""
    layer = {"type": "layer", "adds": adds or {}, "deletes": sorted(deletes)}
    return put(layer)


def blob_size(d: str) -> int:
    """Bytes this blob occupies in the store (our stand-in for tarball size)."""
    return len(json.dumps(BLOBS[d], sort_keys=True))


BASE_FILES = {   # a miniature python:3.12-slim
    "/usr/bin/python3": "<the python 3.12 interpreter binary>",
    "/usr/lib/libc.so.6": "<glibc — the C library scientific wheels link against>",
    "/etc/os-release": 'PRETTY_NAME="Debian GNU/Linux 12 (slim)"',
    "/etc/motd": "welcome to this machine",
}

base_layer = make_layer(BASE_FILES)
deps_layer = make_layer({"/usr/lib/python3.12/site-packages/fastapi/__init__.py": "fastapi 0.111 code"})
code_layer = make_layer({"/app/main.py": "def home(): return 'hello'"})
tidy_layer = make_layer(deletes={"/etc/motd"})          # a WHITEOUT — deletion as a diff

rows = [{"layer": name, "digest": d[7:], "size (B)": blob_size(d),
         "adds": len(BLOBS[d]["adds"]), "whiteouts": len(BLOBS[d]["deletes"])}
        for name, d in [("base", base_layer), ("deps", deps_layer),
                        ("code", code_layer), ("tidy (rm motd)", tidy_layer)]]
print(pd.DataFrame(rows).to_string(index=False))

         layer       digest  size (B)  adds  whiteouts
          base 1e0311ade7b1       292     4          0
          deps a6e9500bbf9a       121     1          0
          code 7f4fb05803c2        88     1          0
tidy (rm motd) 9f83a8449758        55     0          1


Look at the `tidy` layer: deleting a file is itself a *positive* artifact — a few bytes recording a whiteout. Nothing was reclaimed anywhere; `/etc/motd` still sits, at full size, inside `base`'s diff. That is the entire "my `RUN rm -rf` didn't shrink the image" mystery from [images-and-layers.md §2](images-and-layers.md), reproduced in four rows of a table.

## 3. An image = manifest + config + layers

Bundle time. Per the OCI (Open Container Initiative) format, an **image** is just two small JSON documents pointing (by digest) at the layers:

- the **config** — runtime metadata: `Cmd`, `Env`, the ordered layer list;
- the **manifest** — the table of contents: config digest + layer digests.

The real versions of the commands below are dissected in [images-and-layers.md §6](images-and-layers.md):

```bash
docker buildx imagetools inspect python:3.12-slim --raw   # the manifest JSON
docker image inspect python:3.12-slim                     # the config, locally
```

In [5]:
def make_image(layers: list[str], cmd: list[str], env: list[str] = ()) -> str:
    """Assemble config + manifest around a layer stack. Returns the MANIFEST digest."""
    config = {"type": "config", "Cmd": list(cmd), "Env": list(env), "rootfs": list(layers)}
    manifest = {"type": "manifest", "config": put(config), "layers": list(layers)}
    return put(manifest)


def image_bytes(manifest_digest: str) -> int:
    """Total store bytes an image references: manifest + config + every layer."""
    m = BLOBS[manifest_digest]
    return blob_size(manifest_digest) + blob_size(m["config"]) + sum(map(blob_size, m["layers"]))


image_v1 = make_image([base_layer, deps_layer, code_layer],
                      cmd=["python", "/app/main.py"], env=["PYTHONUNBUFFERED=1"])
TAGS["myapp:latest"] = image_v1

print("myapp:latest ->", image_v1, f"({image_bytes(image_v1)} bytes total)\n")
print(json.dumps(BLOBS[image_v1], indent=2))

myapp:latest -> sha256:ddb38edfa744 (800 bytes total)

{
  "type": "manifest",
  "config": "sha256:1e720d856061",
  "layers": [
    "sha256:1e0311ade7b1",
    "sha256:a6e9500bbf9a",
    "sha256:7f4fb05803c2"
  ]
}


That printout *is* the shape of real `docker buildx imagetools inspect --raw` output: a manifest that names a config and an ordered layer stack, everything by digest. Note the cascade: the manifest contains the layer digests, so the manifest's own digest transitively fingerprints every byte of the image — change anything, anywhere, and `myapp`'s digest changes. One hash pins the whole artifact.

## 4. The build cache, replayed

Now let's *build* images the way `docker build` does — and watch its cache work. Recall the mechanics from [images-and-layers.md §7](images-and-layers.md). For each instruction, the builder computes a **cache key** from:

1. the **parent** layer's key (a miss invalidates everything below it — the cascade),
2. the **instruction text**, verbatim,
3. for `COPY`: the **content digest** of the copied files (content, *not* timestamps).

We'll build this Dockerfile — the canonical ordering from [Module 14](../14_cicd/docker.md), requirements before code:

In [6]:
INSTRUCTIONS = [
    ("FROM", "python:3.12-slim"),
    ("ENV",  "PYTHONUNBUFFERED=1"),
    ("COPY", "requirements.txt ."),
    ("RUN",  "pip install -r requirements.txt"),
    ("COPY", "app ./app"),
    ("CMD",  '["python", "-m", "app"]'),
]

context_v1 = {   # the build context: what `docker build .` can see
    "requirements.txt": "fastapi==0.111\nuvicorn==0.30\n",
    "app": "def home(): return 'hello'",
}


def layer_key(parent: str, ins: tuple, context: dict) -> str:
    """The cache key: parent chain + instruction text (+ copied-file content)."""
    instruction, args = ins
    material = parent + "|" + instruction + " " + args
    if instruction == "COPY":
        src = args.split()[0]
        material += "|" + digest({src: context[src]})     # CONTENT, not timestamps
    return "sha256:" + hashlib.sha256(material.encode()).hexdigest()[:12]


def layer_for(ins: tuple, context: dict) -> str:
    """Execute one instruction -> the layer (diff) it produces."""
    instruction, args = ins
    if instruction == "FROM":
        return make_layer(BASE_FILES)                     # the base image's files
    if instruction == "COPY":
        src, dst = args.split()
        path = "/app/" + (src if dst == "." else dst.removeprefix("./"))
        return make_layer({path: context[src]})
    if instruction == "RUN":                              # "install" the pinned deps
        return make_layer({"/usr/lib/python3.12/site-packages/": "installed from: " + context["requirements.txt"]})
    return make_layer({})                                 # ENV / CMD: metadata-only, empty diff


def build(instructions, context, cache: dict) -> tuple[list, list]:
    """Simulate `docker build`: reuse cached layers until the first change."""
    parent, parent_rebuilt, rows, layers = "scratch", False, [], []
    for ins in instructions:
        key = layer_key(parent, ins, context)
        hit = key in cache and not parent_rebuilt         # <- the one rule
        if not hit:
            cache[key] = layer_for(ins, context)
            parent_rebuilt = True
        layers.append(cache[key])
        rows.append({"step": f"{ins[0]} {ins[1]}"[:44], "status": "CACHED ✅" if hit else "REBUILT 🔨"})
        parent = key
    return rows, layers


CACHE: dict[str, str] = {}
cold, layers_v1 = build(INSTRUCTIONS, context_v1, CACHE)
print("BUILD 1 — cold cache:")
print(pd.DataFrame(cold).to_string(index=False))

BUILD 1 — cold cache:
                               step    status
              FROM python:3.12-slim REBUILT 🔨
             ENV PYTHONUNBUFFERED=1 REBUILT 🔨
            COPY requirements.txt . REBUILT 🔨
RUN pip install -r requirements.txt REBUILT 🔨
                     COPY app ./app REBUILT 🔨
        CMD ["python", "-m", "app"] REBUILT 🔨


In [7]:
# BUILD 2 — edit only the app code (the everyday case)
context_edit_code = dict(context_v1, app="def home(): return 'hello, v2'")
warm, _ = build(INSTRUCTIONS, context_edit_code, CACHE)
print("BUILD 2 — after editing app code only:")
print(pd.DataFrame(warm).to_string(index=False))

# BUILD 3 — add a dependency (the expensive case)
context_edit_deps = dict(context_v1, **{"requirements.txt": "fastapi==0.111\nuvicorn==0.30\nrich==13.7\n"})
deps, _ = build(INSTRUCTIONS, context_edit_deps, CACHE)
print("\nBUILD 3 — after adding one line to requirements.txt:")
print(pd.DataFrame(deps).to_string(index=False))

BUILD 2 — after editing app code only:
                               step    status
              FROM python:3.12-slim  CACHED ✅
             ENV PYTHONUNBUFFERED=1  CACHED ✅
            COPY requirements.txt .  CACHED ✅
RUN pip install -r requirements.txt  CACHED ✅
                     COPY app ./app REBUILT 🔨
        CMD ["python", "-m", "app"] REBUILT 🔨

BUILD 3 — after adding one line to requirements.txt:
                               step    status
              FROM python:3.12-slim  CACHED ✅
             ENV PYTHONUNBUFFERED=1  CACHED ✅
            COPY requirements.txt . REBUILT 🔨
RUN pip install -r requirements.txt REBUILT 🔨
                     COPY app ./app REBUILT 🔨
        CMD ["python", "-m", "app"] REBUILT 🔨


Read the two tables against [images-and-layers.md §7](images-and-layers.md):

- **BUILD 2:** everything through the expensive `RUN pip install` is **CACHED**; only the `COPY app` layer and the metadata layer after it rebuild. Seconds, not minutes — this is why the lockfile-before-code ordering exists.
- **BUILD 3:** the `COPY requirements.txt` key changed (its *content* hash changed), so it and **every layer after it** rebuilt — including the pip install. The cascade never skips: a rebuilt parent forces a rebuild below, even for instructions whose own text and inputs are unchanged.

For a web app that cascade costs three minutes. For a 5 GB PyTorch dependency layer it costs twenty — which is why [dockerfiles-for-data-science.md §3](dockerfiles-for-data-science.md) adds BuildKit cache mounts on top of this exact mechanism.

---

### ✋ Quick exercise (~2 min) — Find where the cascade starts

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Write `first_invalidated(rows)` returning the `step` string of the **first** rebuilt layer (or `None` if the whole build was cached). Then check:

1. Where does the cascade start in BUILD 2 (`warm`)?
2. Where in BUILD 3 (`deps`)?

In [8]:
# ✍️ Your turn 👇
def first_invalidated(rows: list[dict]) -> str | None:
    # the "step" of the first row whose status ends with 🔨
    ...

# print("BUILD 2 cascade starts at:", first_invalidated(warm))
# print("BUILD 3 cascade starts at:", first_invalidated(deps))

<details>
<summary>✅ <b>Solution</b></summary>

```python
def first_invalidated(rows: list[dict]) -> str | None:
    return next((r["step"] for r in rows if r["status"].endswith("🔨")), None)


print("BUILD 2 cascade starts at:", first_invalidated(warm))   # COPY app ./app
print("BUILD 3 cascade starts at:", first_invalidated(deps))   # COPY requirements.txt .
```

The cascade's *starting row* is the whole cost model of a Dockerfile: everything below it re-runs. Optimizing a Dockerfile ≈ pushing the likely first-invalidated row as far down as possible.
</details>

## 5. OverlayFS — composing layers into one filesystem

Layers are storage; a running container needs a single, *writable* filesystem. On Linux, **OverlayFS** unions the read-only layers (**`lowerdir`s**) under one writable **`upperdir`** — the container layer — and mounts the composed view at **`merged`**. The real thing, from [images-and-layers.md §5](images-and-layers.md):

```bash
mkdir lower upper work merged
sudo mount -t overlay overlay -o lowerdir=lower,upperdir=upper,workdir=work merged
# merged/ now shows lower's files; writes land in upper/; deletes become whiteouts
```

The lookup rule: **upper wins; then lowers, top-down; a whiteout ends the search.** Modifying a lower file first **copies it up** — in full — into upper. Here is the whole thing as a class:

In [9]:
WHITEOUT = "<whiteout>"      # stand-in for OverlayFS's 0:0 character device / .wh. tar entry


class Overlay:
    """Toy OverlayFS: a writable upper dict over a stack of read-only lower dicts."""

    def __init__(self, lowers: list[dict]):
        self.lowers = list(lowers)          # index 0 = TOPMOST lower (last image layer)
        self.upper: dict[str, str] = {}     # the container layer — starts empty

    def read(self, path: str) -> str | None:
        for layer in [self.upper] + self.lowers:      # top-down; first match wins
            if path in layer:
                v = layer[path]
                return None if v == WHITEOUT else v   # whiteout: file is "deleted"
        return None

    def write(self, path: str, content: str) -> None:
        self.upper[path] = content                     # ALL writes land in upper

    def append(self, path: str, extra: str) -> None:   # modify = COPY-UP first
        current = self.read(path) or ""
        self.upper[path] = current + extra             # the FULL file now sits in upper

    def delete(self, path: str) -> None:
        self.upper[path] = WHITEOUT                    # deletion = a marker, not a removal

    def ls(self) -> list[str]:
        names = set(self.upper) | {p for layer in self.lowers for p in layer}
        return sorted(p for p in names if self.read(p) is not None)


def layer_dir(layer_digest: str) -> dict:
    """Unpack a stored layer into a lower directory (whiteouts included)."""
    layer = BLOBS[layer_digest]
    d = dict(layer["adds"])
    for p in layer["deletes"]:
        d[p] = WHITEOUT
    return d


# docker run myapp  ->  mount the image's layers (reversed: last layer on top) + fresh upper
lowers = [layer_dir(d) for d in reversed(BLOBS[image_v1]["layers"])]
c1 = Overlay(lowers)

print("read from base layer:  ", c1.read("/etc/os-release"))
print("read from code layer:  ", c1.read("/app/main.py"))

c1.write("/tmp/run.log", "starting up...")              # new file -> upper
c1.append("/app/main.py", "  # hotfix")                 # modify -> copy-up into upper
c1.delete("/etc/motd")                                  # delete -> whiteout in upper

print("after edits, merged view of /app/main.py:", c1.read("/app/main.py"))
print("after delete, /etc/motd:", c1.read("/etc/motd"))
print("upper (container layer) now holds:", sorted(c1.upper))
print("the image's code layer is UNTOUCHED:", BLOBS[code_layer]["adds"])

read from base layer:   PRETTY_NAME="Debian GNU/Linux 12 (slim)"
read from code layer:   def home(): return 'hello'
after edits, merged view of /app/main.py: def home(): return 'hello'  # hotfix
after delete, /etc/motd: None
upper (container layer) now holds: ['/app/main.py', '/etc/motd', '/tmp/run.log']
the image's code layer is UNTOUCHED: {'/app/main.py': "def home(): return 'hello'"}


Three things just happened that explain three real-world behaviors:

- **The image never changed.** Every "edit" landed in `upper`. Run `docker diff <container>` on a real container and you are literally listing its `upper` — `A` added, `C` copied-up/changed, `D` whiteout.
- **Copy-up moved the whole file.** `append` had to place the *complete* `main.py` into upper to change one line. Harmless at 30 bytes — ruinous at 2 GB, which is why write-heavy data lives on volumes ([volumes-and-networking.md §1](volumes-and-networking.md)). Watch the cost:

In [10]:
# Copy-up cost, made visible: a "2 GB" model file in a lower layer (simulated at 2 MB)
big_lower = {"/model/weights.bin": "W" * 2_000_000}
c_model = Overlay([big_lower])

print(f"before: upper holds {sum(map(len, c_model.upper.values()))} bytes")
c_model.append("/model/weights.bin", "!")               # append ONE byte...
print(f"after appending 1 byte: upper holds {sum(map(len, c_model.upper.values())):,} bytes  <- full copy-up!")

# ...and the third behavior: many containers, one image.
c2, c3 = Overlay(lowers), Overlay(lowers)               # docker run x2 more
c2.write("/tmp/run.log", "container 2 log")
c3.write("/tmp/run.log", "container 3 log")

shared = sum(blob_size(d) for d in BLOBS[image_v1]["layers"])
print(f"\nshared read-only image layers: {shared} bytes — stored ONCE for all 3 containers")
for name, c in [("c1", c1), ("c2", c2), ("c3", c3)]:
    private = sum(len(v) for v in c.upper.values() if v != WHITEOUT)
    print(f"  {name}: private container layer = {private} bytes")

del c2   # docker rm c2 — the upper dict vanishes; the image layers don't even notice

before: upper holds 0 bytes
after appending 1 byte: upper holds 2,000,001 bytes  <- full copy-up!

shared read-only image layers: 501 bytes — stored ONCE for all 3 containers
  c1: private container layer = 50 bytes
  c2: private container layer = 15 bytes
  c3: private container layer = 15 bytes


---

### ✋ Quick exercise (~3 min) — `docker diff`, reimplemented

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Write `docker_diff(ov)` returning a sorted list of `(flag, path)` tuples describing the container layer, using real `docker diff` flags:

- `"A"` — path exists in upper but in **no** lower (added),
- `"C"` — path exists in upper **and** in some lower (changed / copied-up),
- `"D"` — path is a whiteout in upper (deleted).

Run it on `c1`; you should see `/tmp/run.log` as `A`, `/app/main.py` as `C`, `/etc/motd` as `D`.

In [11]:
# ✍️ Your turn 👇
def docker_diff(ov: Overlay) -> list[tuple[str, str]]:
    # walk ov.upper; classify each path as "A", "C", or "D"
    ...

# for flag, path in docker_diff(c1):
#     print(flag, path)

<details>
<summary>✅ <b>Solution</b></summary>

```python
def docker_diff(ov: Overlay) -> list[tuple[str, str]]:
    out = []
    for path, v in ov.upper.items():
        in_lower = any(path in layer for layer in ov.lowers)
        flag = "D" if v == WHITEOUT else ("C" if in_lower else "A")
        out.append((flag, path))
    return sorted(out, key=lambda t: t[1])


for flag, path in docker_diff(c1):
    print(flag, path)
# C /app/main.py     (copied up)
# D /etc/motd        (whiteout)
# A /tmp/run.log     (new)
```

This is not an analogy — real `docker diff` walks the real `upperdir` and prints exactly these three flags. You've reimplemented a Docker subcommand in eight lines.
</details>

## 6. Namespaces — a private process table per container

Filesystem done; now the *isolation* half, from [container-internals.md §3](container-internals.md). A **pid namespace** gives a process group its own process table: inside, the main process is **PID 1** and neighbors are invisible; outside, the host sees the same processes under ordinary host PIDs. The real commands this section mirrors:

```bash
sudo unshare --pid --fork --mount-proc bash   # a shell that is PID 1 in a fresh namespace
docker top <container>                        # the container-PID -> host-PID mapping
ps aux                                        # inside a container: just your processes
```

We model the host as one global process table, and a namespace as a **mapping** from container PIDs to host PIDs:

In [12]:
import itertools


class PidNamespace:
    _ids = itertools.count(1)

    def __init__(self):
        self.id = f"pidns-{next(PidNamespace._ids)}"
        self.cmap: dict[int, int] = {}          # host pid -> container pid
        self._pids = itertools.count(1)         # container pids start at 1

    def register(self, host_pid: int) -> int:
        self.cmap[host_pid] = next(self._pids)
        return self.cmap[host_pid]


class Host:
    """One machine: ONE real process table. Namespaces are views, not copies."""

    def __init__(self, hostname: str):
        self.hostname = hostname
        self.procs: dict[int, dict] = {}        # host pid -> {"cmd", "ns"}
        self._pids = itertools.count(100)       # host pids: ordinary numbers

    def spawn(self, cmd: str, ns: PidNamespace | None = None) -> int:
        hpid = next(self._pids)
        self.procs[hpid] = {"cmd": cmd, "ns": ns}
        if ns is not None:
            ns.register(hpid)
        return hpid


def ps(host: Host, ns: PidNamespace | None = None) -> None:
    """ns=None -> the HOST's view (all processes). ns=... -> the view INSIDE that namespace."""
    if ns is None:
        print(f"  PID   NS        CMD                 ({host.hostname}: host view)")
        for hpid, p in sorted(host.procs.items()):
            print(f"  {hpid:<5} {p['ns'].id if p['ns'] else '-':<9} {p['cmd']}")
    else:
        print(f"  PID  CMD                 (view inside {ns.id})")
        for hpid, cpid in sorted(ns.cmap.items(), key=lambda kv: kv[1]):
            print(f"  {cpid:<4} {host.procs[hpid]['cmd']}")


host = Host("gpu-server")
host.spawn("systemd"); host.spawn("sshd"); host.spawn("dockerd")


def docker_run(host: Host, name: str, cmd: str) -> dict:
    """The §2-of-container-internals chain, condensed: fresh namespace + first process."""
    ns = PidNamespace()
    return {"name": name, "ns": ns, "main": host.spawn(cmd, ns=ns)}


api   = docker_run(host, "api",   "python -m uvicorn app:app")
train = docker_run(host, "train", "python train.py")

ps(host, api["ns"]);   print()
ps(host, train["ns"]); print()
ps(host)

  PID  CMD                 (view inside pidns-1)
  1    python -m uvicorn app:app

  PID  CMD                 (view inside pidns-2)
  1    python train.py

  PID   NS        CMD                 (gpu-server: host view)
  100   -         systemd
  101   -         sshd
  102   -         dockerd
  103   pidns-1   python -m uvicorn app:app
  104   pidns-2   python train.py


Both containers believe they are **PID 1** — each sees a one-line process table. The host's view tells the truth: the same two processes are ordinary PIDs 103 and 104, tagged with the namespace they belong to. Nothing was virtualized; the kernel (our `Host`) just filters what each viewer is shown.

And `docker exec`? It is **not** a login to some machine — it spawns a *new* process and **joins it into the existing namespace** (the `setns` syscall):

In [13]:
def docker_exec(host: Host, container: dict, cmd: str) -> int:
    """setns(): a NEW process, joined into an EXISTING container's namespace."""
    return host.spawn(cmd, ns=container["ns"])


docker_exec(host, api, "bash")      # you, debugging

ps(host, api["ns"])
print("\nThe exec'd bash is container-PID 2 — a guest in api's namespace,")
print("sharing its process view, filesystem, and fate: kill the container, kill the shell.")

  PID  CMD                 (view inside pidns-1)
  1    python -m uvicorn app:app
  2    bash

The exec'd bash is container-PID 2 — a guest in api's namespace,
sharing its process view, filesystem, and fate: kill the container, kill the shell.


---

### ✋ Quick exercise (~2 min) — Reimplement `docker top`

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

`docker top <container>` shows the **mapping** both sides of the one-way glass: each process's container PID *and* its host PID. Write `docker_top(host, container)` returning a list of `(container_pid, host_pid, cmd)` tuples, sorted by container PID, and run it on `api` — you should see uvicorn as `(1, 103, …)` and your exec'd bash as `(2, …)`.

In [14]:
# ✍️ Your turn 👇
def docker_top(host: Host, container: dict) -> list[tuple[int, int, str]]:
    # walk container["ns"].cmap  (host pid -> container pid)
    ...

# for cpid, hpid, cmd in docker_top(host, api):
#     print(f"container PID {cpid} = host PID {hpid}  ({cmd})")

<details>
<summary>✅ <b>Solution</b></summary>

```python
def docker_top(host: Host, container: dict) -> list[tuple[int, int, str]]:
    ns = container["ns"]
    return sorted(((cpid, hpid, host.procs[hpid]["cmd"])
                   for hpid, cpid in ns.cmap.items()))


for cpid, hpid, cmd in docker_top(host, api):
    print(f"container PID {cpid} = host PID {hpid}  ({cmd})")
```

The mapping is the demystifier: "in a container" is a *property of a process* (which namespace objects its `/proc/<pid>/ns/` entries point at), not a place. The host can always see through; the container can never see out.
</details>

## 7. cgroups — the memory meter that kills at 137

Namespaces limit what a process can *see*; **cgroups** limit what it can *use* ([container-internals.md §4](container-internals.md)). The flag, the file, and the fingerprint:

```bash
docker run --memory=4g my-training-image      # -> memory.max = 4294967296 in the cgroup
docker inspect --format '{{.State.OOMKilled}} {{.State.ExitCode}}' <c>   # true 137
```

The crucial behavior: **memory limits don't throttle — they kill.** Cross `memory.max` and the kernel's OOM (out-of-memory) killer sends SIGKILL: exit code 128 + 9 = **137**, and no Python traceback, because the process died mid-instruction. Simulated:

In [15]:
class Cgroup:
    """A memory controller: meter every allocation, kill on breach."""

    def __init__(self, memory_max_mb: int):
        self.memory_max = memory_max_mb
        self.current = 0
        self.oom_killed = False

    def alloc(self, mb: int) -> bool:
        if self.current + mb > self.memory_max:
            self.oom_killed = True              # the OOM killer fires: SIGKILL, no cleanup
            return False
        self.current += mb
        return True


# docker run --memory=4g train  ... where each batch "leaks" 700 MB
cg, exit_code = Cgroup(memory_max_mb=4096), 0
for batch in itertools.count(1):
    if not cg.alloc(700):
        print(f"batch {batch}: +700 MB would exceed memory.max -> OOM killer -> SIGKILL")
        exit_code = 137
        break
    print(f"batch {batch}: +700 MB   (usage {cg.current}/{cg.memory_max} MB)")

print("\ndocker inspect says:")
print(json.dumps({"State": {"OOMKilled": cg.oom_killed, "ExitCode": exit_code}}, indent=2))

batch 1: +700 MB   (usage 700/4096 MB)
batch 2: +700 MB   (usage 1400/4096 MB)
batch 3: +700 MB   (usage 2100/4096 MB)
batch 4: +700 MB   (usage 2800/4096 MB)
batch 5: +700 MB   (usage 3500/4096 MB)
batch 6: +700 MB would exceed memory.max -> OOM killer -> SIGKILL

docker inspect says:
{
  "State": {
    "OOMKilled": true,
    "ExitCode": 137
  }
}


Five successful batches, then silence — the exact signature of the "my training container vanished overnight" incident. When you meet exit 137 in the wild: check `OOMKilled`, then either raise `--memory`, shrink the working set (batch size, streaming loading), or — on a Mac — raise Docker Desktop's **VM** memory, the hidden ceiling above every container ([container-internals.md §6](container-internals.md)).

### Is real Docker around?

Nothing above needed it — but if Docker is installed, every simulation you just ran has a one-line reality check. This cell only reports; it runs nothing.

In [16]:
import shutil, subprocess

if shutil.which("docker"):
    out = subprocess.run(["docker", "--version"], capture_output=True, text=True)
    print("Docker is installed:", out.stdout.strip())
    print("Try the real versions of today's toys:")
    print("  docker history python:3.12-slim          # §2-3: the layer stack")
    print("  docker build . && edit && rebuild        # §4: watch CACHED lines")
    print("  docker run -d --name c alpine sleep 999 && docker exec c touch /x && docker diff c")
    print("  docker top c                             # §6: the PID mapping")
    print("  docker run --rm --memory=64m python:3.12-slim python -c 'bytearray(10**9)'; echo $?")
else:
    print("No Docker here — completely fine: this lab is self-contained.")
    print("Keep the commands above for the day you have a Docker machine handy.")

No Docker here — completely fine: this lab is self-contained.
Keep the commands above for the day you have a Docker machine handy.


## 🧪 Practice exercises

Work these after the walk-through. Each ships a collapsible solution — try first, then check.

### Exercise 1 — ⭐ A `docker pull`, layer by layer

Pulling an image means: fetch the manifest, then fetch **only the blobs you don't already have** — which is why pull output mixes `Already exists` and `Pull complete` lines. Write `pull(manifest_digest, local)` that walks an image's config + layers against a `local` set of digests, prints one status line per blob, adds what it "downloads" to `local`, and returns total bytes downloaded.

Scenario: your laptop already has the base layer (you pulled some other slim-based image yesterday): `local = {base_layer}`. Pull `myapp:latest` — how many bytes does the shared base save?

In [17]:
# your code here
local = {base_layer}
# def pull(manifest_digest: str, local: set) -> int: ...
# downloaded = pull(TAGS["myapp:latest"], local)

<details>
<summary>✅ <b>Solution</b></summary>

```python
def pull(manifest_digest: str, local: set) -> int:
    m = BLOBS[manifest_digest]
    downloaded = 0
    for d in [m["config"], *m["layers"]]:
        if d in local:
            print(f"{d[7:]}: Already exists")
        else:
            print(f"{d[7:]}: Pull complete ({blob_size(d)} B)")
            local.add(d)
            downloaded += blob_size(d)
    return downloaded


local = {base_layer}
downloaded = pull(TAGS["myapp:latest"], local)
print(f"\ndownloaded {downloaded} B; the shared base layer saved {blob_size(base_layer)} B")
```

The base layer — by far the biggest blob — transfers zero bytes, because content addressing lets the client *prove* it already has those exact bytes. Scale the toy up and this is why pulling your tenth `python:3.12-slim`-based image takes seconds, not minutes.
</details>

### Exercise 2 — ⭐ Deduplication across images

Build `image_v2`: same base and deps layers, but a changed `/app/main.py` (a new code layer), tagged `myapp:v2`. Then compute:

1. **naive total** — `image_bytes(v1) + image_bytes(v2)` (what two unrelated tarballs would cost),
2. **actual store bytes** — the sum of `blob_size` over the **set** of digests both images reference,
3. the percentage saved by content addressing.

In [18]:
# your code here
# code_layer_v2 = make_layer({...})
# image_v2 = make_image([...], cmd=["python", "/app/main.py"])
# TAGS["myapp:v2"] = image_v2

<details>
<summary>✅ <b>Solution</b></summary>

```python
code_layer_v2 = make_layer({"/app/main.py": "def home(): return 'hello, v2'"})
image_v2 = make_image([base_layer, deps_layer, code_layer_v2], cmd=["python", "/app/main.py"])
TAGS["myapp:v2"] = image_v2

def referenced(mdig):
    m = BLOBS[mdig]
    return {mdig, m["config"], *m["layers"]}

naive  = image_bytes(image_v1) + image_bytes(image_v2)
actual = sum(blob_size(d) for d in referenced(image_v1) | referenced(image_v2))
print(f"naive: {naive} B   actual: {actual} B   saved: {1 - actual/naive:.0%}")
```

Only the tiny new code layer, config, and manifest are new bytes — base and deps are the *same digests*, stored once. This is the storage story of every CI pipeline that pushes daily: a hundred image versions, one copy of the heavy layers.
</details>

### Exercise 3 — ⭐⭐ The `rm` that didn't shrink

Reproduce the classic with our machinery. Build two images on top of `base_layer`:

- **`bloated`** — layer 2 adds `/var/cache/big.bin` (`"B" * 50_000`), layer 3 *deletes* it (a whiteout layer);
- **`clean`** — a single layer 2 that never adds the file at all (the "same-instruction cleanup").

Show that (1) in the merged view of `bloated`, the file is invisible; (2) `image_bytes(bloated)` is still ~50 KB larger than `image_bytes(clean)`.

In [19]:
# your code here
# big_layer = make_layer({...}); rm_layer = make_layer(deletes={...})
# bloated = make_image([...], cmd=["python"]) ; clean = make_image([...], cmd=["python"])

<details>
<summary>✅ <b>Solution</b></summary>

```python
big_layer = make_layer({"/var/cache/big.bin": "B" * 50_000})
rm_layer  = make_layer(deletes={"/var/cache/big.bin"})

bloated = make_image([base_layer, big_layer, rm_layer], cmd=["python"])
clean   = make_image([base_layer, make_layer({})],       cmd=["python"])

ov = Overlay([layer_dir(d) for d in reversed(BLOBS[bloated]["layers"])])
print("merged view of big.bin:", ov.read("/var/cache/big.bin"))          # None — invisible
print(f"bloated: {image_bytes(bloated):,} B   clean: {image_bytes(clean):,} B")
print(f"difference: ~{image_bytes(bloated) - image_bytes(clean):,} B still shipped, invisibly")
```

The whiteout in `rm_layer` hides the file from every container — but `big_layer`, with all 50 000 bytes, is still a referenced blob that every pull downloads and every disk stores. The only fixes are structural: never snapshot the file (same-instruction cleanup, cache mounts) or drop the layer entirely (multi-stage builds) — [dockerfiles-for-data-science.md](dockerfiles-for-data-science.md).
</details>

### Exercise 4 — ⭐⭐ Debug me 🐞

A teammate wrote a bulk reader for the merged view. It crashes on `c1`. **Run it, read the traceback, and fix it** so it returns the merged contents, skipping deleted/missing files.

*Hint: what does `Overlay.read` return for a whiteout — and what does `None` not have?*

In [20]:
# 🐞 Buggy — run it, watch it break, then fix it.
def read_many(ov: Overlay, paths: list[str]) -> dict:
    return {p: ov.read(p).strip() for p in paths}      # bug: read() may return None


print(read_many(c1, ["/app/main.py", "/etc/motd", "/tmp/run.log"]))

AttributeError: 'NoneType' object has no attribute 'strip'

<details>
<summary>✅ <b>Solution</b></summary>

```python
def read_many(ov: Overlay, paths: list[str]) -> dict:
    out = {}
    for p in paths:
        content = ov.read(p)
        if content is not None:          # whiteout-deleted or absent -> skip
            out[p] = content.strip()
    return out


print(read_many(c1, ["/app/main.py", "/etc/motd", "/tmp/run.log"]))
# /etc/motd is missing from the result: c1 deleted it (whiteout) in §5
```

`/etc/motd` *exists in a lower layer* but is whited-out in upper, so `read` correctly reports `None` — "file not found" in the merged view. Real code touching container filesystems must treat "present below but deleted above" as gone; that's not an edge case, it's the semantics.
</details>

## 🧠 Stretch exercises

Harder, open-ended. Solutions sketch the approach rather than hand you the code.

### Stretch exercise A — ⭐⭐⭐ Simulate a multi-stage build

Extend `build()` to support `("FROM", "... AS builder")` and `("COPY", "--from=builder /opt/venv /opt/venv")`: run the instruction list as *stages* (each `FROM` starts a new layer stack), and let a `--from=` COPY reach into an earlier stage's built filesystem. Then show the punchline with `image_bytes`: a builder stage that installs a 40 KB "toolchain" layer plus a 5 KB "venv" layer, and a final image that carries **only** the venv — the toolchain bytes appear in no referenced blob of the final manifest. That's [dockerfiles-for-data-science.md §4](dockerfiles-for-data-science.md) in miniature: multi-stage is the one pattern that *removes* layers instead of hiding them.

### Stretch exercise B — ⭐⭐⭐ Add user namespaces to the host model

Give `PidNamespace` an optional `uid_map` (e.g. `{0: 100000}`: container UID 0 → host UID 100000) and store an owner UID on each spawned process. Extend `ps` so the container view shows `root` for its PID 1 while the host view shows the same process owned by `100000` — an unprivileged nobody. You will have reproduced rootless containers' core trick ([container-internals.md §3](container-internals.md), Podman in [ecosystem.md §1](ecosystem.md)): *root inside, nobody outside* is just one more mapping table.

### Stretch exercise C — ⭐⭐⭐ Simulate a BuildKit cache mount

Model the §4 pain: give `layer_for`'s RUN branch a "download" step that reports how many bytes of wheels it fetches. Then add an optional `run_cache: dict` (the `--mount=type=cache` directory): downloads consult it first, fill it on miss — but its contents **never** enter `make_layer`'s diff. Rebuild after adding one package and compare bytes downloaded with vs without the cache, and confirm `image_bytes` is *identical* in both cases: faster builds, zero image weight — the whole value proposition of `RUN --mount=type=cache` ([dockerfiles-for-data-science.md §3](dockerfiles-for-data-science.md)).

## 🎁 Bonus mini-project — `minidocker`: the whole engine in ~80 lines

Everything you built today composes into one object. Write a `MiniDocker` class that ties the sections together end-to-end:

1. **`build(dockerfile_text, context) -> ref`** — parse a real(istic) Dockerfile string (`FROM` / `ENV` / `COPY` / `RUN` / `CMD`, one per line, `#` comments) into instruction tuples, run them through §4's cached `build()`, assemble a §3 manifest, and return its digest. Support `-t` tagging into `TAGS`.
2. **`run(ref, name, memory_mb=…) -> container`** — resolve tag-or-digest (§1's `resolve`!), mount a §5 `Overlay` from the image's layers, create a §6 `PidNamespace` + spawn the config's `Cmd` as PID 1 on a shared `Host`, and attach a §7 `Cgroup`.
3. **`exec(name, cmd)`**, **`top(name)`**, **`diff(name)`**, **`ps()`** — thin wrappers over what you already wrote (`docker_exec`, `docker_top`, `docker_diff`).
4. **`rm(name)`** — drop the container: upper layer and namespace go; image blobs stay untouched.

**Acceptance test** (write it first!): build an image twice — second build fully cached; run **two** containers from it — both PID 1, private writable layers, shared layer bytes counted once; `exec` a shell into one and see it in `top`; allocate past the memory limit and observe `OOMKilled: true, ExitCode: 137`; `rm` both and show `BLOBS` unchanged.

<details>
<summary>💡 <b>Hints</b> (open after a real attempt)</summary>

- The Dockerfile parser is five lines: strip comments/blanks, `split(maxsplit=1)` into `(INSTRUCTION, args)`.
- Store containers as `{"name": …, "overlay": …, "ns": …, "cgroup": …, "image": ref}` in a dict keyed by name; `ps()` is one loop over it.
- `run` should *not* copy any layer — that's the point. The overlay's `lowers` are views into `BLOBS` via `layer_dir`.
- Keep a single module-level `Host` (a machine has one kernel); each `run` adds one namespace to it.
- For the OOM test, reuse §7's loop but let `run` return the container so the test can call `container["cgroup"].alloc(...)` directly.
- Done early? Add `pull`/`push` between two `BLOBS`-like stores using Exercise 1's `pull` — you now have the entire image lifecycle.
</details>

## 🧠 Key takeaways

- **Content addressing is the foundation.** Every artifact — layer, config, manifest — is stored and referenced by the SHA-256 of its bytes; tags are just re-pointable names over that store. Digests can't lie, tags can.
- **A layer is a diff, and diffs only add.** Even a deletion is a positive artifact (a whiteout). Files "removed" in later layers still cost storage and bandwidth — only same-instruction cleanup, cache mounts, or multi-stage builds truly remove weight.
- **The build cache is one rule applied per line:** reuse a layer iff its key (parent chain + instruction text + copied-file *content*) is known and every parent hit. The first miss cascades to the bottom — Dockerfile optimization is the art of pushing that first miss down.
- **A container filesystem is a mount, not a copy.** OverlayFS unions read-only lowers under one writable upper; reads search top-down, modifications copy up *whole files*, deletes drop whiteouts. `docker diff` is a walk of the upper dir.
- **A container is a process wearing kernel bookkeeping.** Namespaces filter what it *sees* (its own PID 1 and process table); cgroups meter what it *uses* (exceed `memory.max` → OOM kill → exit 137); `docker exec` merely joins a new process into existing namespaces.
- **None of this is Docker-specific.** You modeled the OCI image format and Linux kernel primitives — the same machinery under Podman, containerd, and Kubernetes ([ecosystem.md](ecosystem.md)).

## ✅ Self-assessment

- [ ] I can explain why two identical files always produce the same digest, and why that makes deduplication and verification free.
- [ ] I can explain the difference between re-pointing a tag and the immutability of a digest, and when each reference belongs in a workflow.
- [ ] I can predict, for any single-file edit, exactly which layers of a Dockerfile rebuild — and where the cascade starts.
- [ ] I can trace a read, a write, a modification, and a deletion through upper/lower dirs, including copy-up costs and whiteouts.
- [ ] I can explain why a file deleted in a later layer still bloats an image, and name the three real fixes.
- [ ] I can describe what `docker run`, `docker exec`, `docker top`, and `docker diff` each do in terms of namespaces, overlays, and cgroups — not magic.

## 🚀 Next step

You've now *built* what the chapters described — which is the durable kind of understanding. Next:

- **[Dockerfiles for data science](dockerfiles-for-data-science.md)** — apply today's mechanics (cache cascades, layer weight, whiteout economics) to the 5-GB-dependency, GPU-shaped images this course actually ships.
- **[Volumes & networking](volumes-and-networking.md)** — where data lives once you know why the writable layer is disposable.
- **[exercises.md](exercises.md)** — the module's consolidated exercise bank; sections 2 and 3 drill today's material with and without Docker.
- **[Module 14's labs](../14_cicd/lab01_docker_and_compose.ipynb)** — the same simulation style pointed at the *pipeline*: Dockerfile parsing, Compose startup order, CI, and deployment.